### 📦 Offline Dependency Installation

- Installs `datasets` and `trl` into a local directory (`/kaggle/working/packages`)
- Prefers **offline packages** (for reproducibility & no internet dependency)
- Falls back to PyPI if offline packages are unavailable
- Appends install path to `sys.path`

✅ Ensures Kaggle-compatible, reproducible environment  
⚠️ Avoids permission issues with global installs

In [1]:
import subprocess, sys, os
from pathlib import Path
def resolve_python_path(target_dir):
    for pth_file in Path(target_dir).glob("*.pth"):
        with pth_file.open() as fp:
            relpath = fp.read()
            rel_pack_path = (pth_file.parent/relpath)
            if rel_pack_path.exists():
                print(f"append {rel_pack_path}")
                sys.path.append(str(rel_pack_path))



offline_dir = "/kaggle/input/nvidia-nemotron-offline-packages/offline_packages"
target_dir = "/kaggle/working/packages"

os.makedirs(target_dir, exist_ok=True)
resolve_python_path("/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/")

if os.path.exists(offline_dir):
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-index",
        "--find-links", offline_dir,
        "--target", target_dir,
        "datasets", "trl"
    ])
    print("Installed from offline packages")


# Add to Python path
sys.path.append(target_dir)
resolve_python_path(target_dir)

import datasets, cutlass

append /kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/nvidia_cutlass_dsl/python_packages


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
torchvision 0.26.0.dev20260324+cu128 requires torch==2.12.0.dev20260324, but you have torch 2.10.0 which is incompatible.
cuda-python 12.9.6 requires cuda-bindings~=12.9.6, but you have cuda-bindings 12.9.4 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.1 which is incompatible.
ydata-profiling 4.18.1 requires scipy<1.17,>=1.8, but you have scipy 1.17.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
goo

Installed from offline packages


### ⚙️ Environment Setup & Libraries

- Enables `expandable_segments` → reduces CUDA memory fragmentation (important for 30B)
- Imports:
  - `torch`, `transformers` → model training
  - `polars` → fast data loading
  - `datasets` → HF dataset wrapper
  - `trl`, `peft` → SFT + LoRA fine-tuning
  - `kagglehub` → model download

✅ Optimized for large-model training on H100

In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import sys, stat, shutil, gc, zipfile
import polars as pl
import torch
import torch.nn.functional as F
import kagglehub
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


### 🛠️ Triton + RMSNorm Fix (Kaggle Hack)

- Replaces `rmsnorm_fn` with a pure PyTorch implementation
  → avoids Triton kernel crashes

- Copies & patches `ptxas` binary for Blackwell compatibility
- Overrides Triton backend paths + permissions

✅ Fixes incompatibilities between:
  - Kaggle runtime
  - Triton compiler
  - Nemotron kernels

⚠️ Hacky but necessary for stable training

In [3]:
# === Triton fixes ===
def _pure_rmsnorm_fn(x, weight, bias=None, z=None, eps=1e-5,
                     group_size=None, norm_before_gate=True, upcast=True):
    dtype = x.dtype
    if upcast: x = x.float()
    variance = x.pow(2).mean(-1, keepdim=True)
    x_normed = x * torch.rsqrt(variance + eps)
    out = x_normed * weight.float()
    if bias is not None: out = out + bias.float()
    if z is not None: out = out * F.silu(z.float())
    return out.to(dtype)

for name, mod in list(sys.modules.items()):
    if hasattr(mod, 'rmsnorm_fn'):
        mod.rmsnorm_fn = _pure_rmsnorm_fn

# WARNING: This patch is strictly for Kaggle's environment.
src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/triton/backends/nvidia/bin/ptxas-blackwell"
dst = "/tmp/ptxas-blackwell"
if os.path.exists(src):
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    import triton.backends.nvidia as nv_backend
    src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
    dst_bin = "/tmp/triton_nvidia_bin"
    shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
    for f in os.listdir(dst_bin):
        fp = os.path.join(dst_bin, f)
        if os.path.isfile(fp):
            os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    os.environ["TRITON_PTXAS_PATH"] = dst
    print("Triton ptxas fix applied.")

Triton ptxas fix applied.


### 📊 Training Config + Dataset

**Hyperparameters:**
- LoRA rank = 32
- Seq length = 2048 → supports reasoning chains
- Batch size = 1 + grad accumulation = 4
- LR = 2e-4, epochs = 2

**Data:**
- Loads Nemotron reasoning dataset
- Subsamples 600 rows (fast experimentation)
- Converts to HuggingFace Dataset

✅ Balanced for:
  - H100 VRAM usage
  - Reasoning fine-tuning

In [4]:
# === Hyperparameters ===
SUBSAMPLE_SIZE = 600
LORA_RANK = 32
MAX_SEQ_LEN = 2048     # Increased from 1024 to support reasoning chains
NUM_EPOCHS = 2         # Increased for better convergence
BATCH_SIZE = 1
GRAD_ACCUM = 4         
LR = 1e-4
OUTPUT_DIR = "/kaggle/working/adapter"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# === Data ===
MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
train_df = pl.read_csv('/kaggle/input/datasets/kienngx/nemotron-30b-competition-trainingdata-cot-labels/final_Nemotron_training_data.csv')
train_df = train_df.sample(n=SUBSAMPLE_SIZE, seed=42)
hf_dataset = Dataset.from_pandas(train_df.to_pandas())

### 💬 Training Prompt Construction

- Converts dataset into chat format:
  - User: problem + instruction (`\\boxed{}`)
  - Assistant: full answer (reasoning + final)

- Uses `tokenizer.apply_chat_template`
  → ensures model-native formatting

- Fallback to manual template if needed

⚠️ Critical assumption:
- `answer` must contain reasoning BEFORE final result

✅ Aligns training with inference behavior

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def build_training_text(example):
    prompt = example["prompt"]
    answer = example["answer"] 
    cot = example["generated_cot"] # Extract the reasoning chain
    
    user_msg = prompt + "\nPut your final answer inside \\boxed{}."
    
    # Combine the CoT with the final answer
    assistant_msg = f"{cot}\n\n\\boxed{{{answer}}}" 
    
    try:
        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_msg},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    except Exception:
        text = (
            f"<|im_start|>user\n{user_msg}<|im_end|>\n"
            f"<|im_start|>assistant\n{assistant_msg}<|im_end|>"
        )
    return {"text": text}

# Apply mapping
hf_dataset = hf_dataset.map(build_training_text, remove_columns=hf_dataset.column_names)

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

### 🧠 Model Loading & LoRA Fine-tuning

**Model:**
- Loads Nemotron-3 Nano 30B (bf16, GPU)
- Enables gradient checkpointing → saves VRAM

**Patching:**
- Disables fast path kernels (stability fix)

**LoRA:**
- Rank = 32, alpha = 32
- Targets all major projection layers
- Reduces trainable params significantly

**Training:**
- Uses `SFTTrainer` (TRL)
- Cosine LR scheduler + warmup
- No checkpoint saving (Kaggle constraint)

✅ Efficient fine-tuning of 30B on single H100

In [6]:
# === Model ===
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, 
    device_map={"": 0}, 
    trust_remote_code=True, 
    dtype=torch.bfloat16,
)
model.gradient_checkpointing_enable()

for name, mod in sys.modules.items():
    if "modeling_nemotron_h" in name:
        mod.is_fast_path_available = False
        print(f"Patched {name}: is_fast_path_available = False")

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=32,
    target_modules="all-linear",
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# === Triton compiler fix ===
import triton.backends.nvidia.compiler as nv_compiler
os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
nv_compiler.get_ptxas_version = lambda arch: "12.0"

# === Training ===
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE, # Optimized
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    logging_steps=5,
    bf16=True,
    max_grad_norm=1.0,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
    packing=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

trainer = SFTTrainer(
    model=model,
    train_dataset=hf_dataset,
    processing_class=tokenizer,
    args=training_args,
)

print("Starting training...")
trainer.train()
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR) # Ensure tokenizer config is saved

print(f"Adapter saved to {OUTPUT_DIR}:")
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(os.path.join(OUTPUT_DIR, f))
    print(f"  {f} ({size/1024:.1f} KB)")

Loading weights:   0%|          | 0/6243 [00:00<?, ?it/s]

Patched transformers_modules._1.modeling_nemotron_h: is_fast_path_available = False


/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:122: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_tensor.py:195: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_scaling_utils.py:90: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_in_graph
/usr/local/lib/python3.12/dist-packages/torchao/float8/float8_linear.py:60: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  @torch._dynamo.allow_

trainable params: 883,873,792 || all params: 32,461,811,136 || trainable%: 2.7228


Adding EOS to train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

...

Truncating train dataset:   0%|          | 0/600 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 11, 'pad_token_id': 11}.


Starting training...


Step,Training Loss
5,10.873795
10,9.749901
15,10.146339
20,8.406679
25,6.488879
30,8.501383
35,6.088332
40,6.232933
45,6.800073
50,8.372046


Adapter saved to /kaggle/working/adapter:
  tokenizer_config.json (0.4 KB)
  chat_template.jinja (10.3 KB)
  adapter_model.safetensors (3454393.7 KB)
  README.md (5.2 KB)
  tokenizer.json (16677.2 KB)
  adapter_config.json (1.1 KB)


### 💾 Save LoRA Adapter

- Saves:
  - `adapter_model.safetensors`
  - `adapter_config.json`
  - tokenizer config

- Prints file sizes for verification

✅ Only LoRA weights saved (NOT full model)
→ lightweight (~MBs instead of GBs)

### 📦 Create Submission ZIP

- Compresses adapter files into `submission.zip`
- Verifies required files exist:
  - `adapter_config.json`
  - `adapter_model.safetensors`

- Raises error if missing (prevents submission failure)

✅ Ready for Kaggle submission
⚠️ Missing files = instant evaluation failure

In [7]:
import zipfile
import os

# Define paths
OUTPUT_DIR = "/kaggle/working/adapter"
zip_path = "/kaggle/working/submission.zip"

print(f"Packaging files from {OUTPUT_DIR}...")

# Create the zip file with compression
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        zf.write(fpath, fname) 

print(f"Created {zip_path} ({os.path.getsize(zip_path) / 1024 / 1024:.1f} MB)")

with zipfile.ZipFile(zip_path, 'r') as zf:
    zip_contents = zf.namelist()
    print(f"Zip Contents: {zip_contents}")
    
    if "adapter_config.json" not in zip_contents:
        raise AssertionError("CRITICAL ERROR: adapter_config.json is missing from the zip. The Kaggle evaluation will fail.")
    if "adapter_model.safetensors" not in zip_contents:
         raise AssertionError("CRITICAL ERROR: adapter_model.safetensors is missing from the zip.")

print("✅ submission.zip is ready! You can now submit this file to the competition.")

Packaging files from /kaggle/working/adapter...
Created /kaggle/working/submission.zip (3105.5 MB)
Zip Contents: ['tokenizer_config.json', 'chat_template.jinja', 'adapter_model.safetensors', 'README.md', 'tokenizer.json', 'adapter_config.json']
✅ submission.zip is ready! You can now submit this file to the competition.
